In [ ]:
#-------Fenwick Tree — Fast Cumulative Sums
"""
A Fenwick Tree, also called a Binary Indexed Tree, 
solves a similar problem to a Segment Tree, but with a different and often simpler structure.
 Imagine daily sales:
    Day:       1   2   3   4   5
    Sales:    10  20  30  40  50
Total sales up to Day 4? -- 10 + 20 + 30 + 40 = 100

And perhaps sales change: Day 3:   30 → 35 . efficiently update the data and calculate totals again.

Index       Stored information
----        -------------------
1           10
2           10 + 20
3           30
4           10 + 20 + 30 + 40
5           50

----------Operation index & -index below is as follows - 
| Sales | Index | Index in binary | `-index` in binary* | `index & -index` | Next index |
| ----: | ----: | --------------: | ------------------: | ---------------: | ---------: |
|    10 |     1 |           `001` |               `111` |        `001` = 1 |          2 |
|    20 |     2 |           `010` |               `110` |        `010` = 2 |          4 |
|    30 |     3 |           `011` |               `101` |        `001` = 1 |          4 |
|    40 |     4 |           `100` |               `100` |        `100` = 4 |          8 |
|    50 |     5 |           `101` |               `011` |        `001` = 1 |          6 |

""" 
sales = [10, 20, 30, 40, 50]
tree = [0] * (len(sales) + 1)                # << -- This is storage similar to Segment tree

def update(index, value):
    while index < len(tree):
        tree[index] += value
        index += index & -index              #<<--- Very important to populate next index
def prefix_sum(index):
    total = 0
    while index > 0:
        total += tree[index]
        index -= index & -index
    return total


# Build the Fenwick Tree
for i in range(len(sales)):
    update(i + 1, sales[i])

# Total sales up to Day 4
answer = prefix_sum(4)
print(answer)

100


In [ ]:
#------------B-Tree ------------one node → multiple values
"""
              [30 | 60]
             /    |     \
        [10 20] [40 50] [70 80 90]

root.children[0]  → left
root.children[1]  → middle
root.children[2]  → right
"""

class BTreeNode:
    def __init__(self, keys=None):     # << -- root.keys = [30, 60]
        self.keys = keys or []
        self.children = []

# Create nodes
root = BTreeNode([30, 60])             #<< - Here we are passing multiple values in list 
left = BTreeNode([10, 20])
middle = BTreeNode([40, 50])
right = BTreeNode([70, 80, 90])

root.children = [left, middle, right]  #<<-- “Attach these three nodes as children of root.”

def print_tree(node, level=0):
    print("   " * level + str(node.keys))
    for child in node.children:
        print_tree(child, level + 1)

print_tree(root)

def search(node, value):
    # Look inside this node
    for i in range(len(node.keys)):
        if value == node.keys[i]:
            return True

        # Value belongs in the child before this key
        if value < node.keys[i]:
            if len(node.children) > i:
                return search(node.children[i], value)
            return False

    # Value is greater than all keys
    if len(node.children) > len(node.keys):
        return search(node.children[-1], value)
    return False

print(search(root, 50))
print(search(root, 75))
print(search(root, 100))

[30, 60]
   [10, 20]
   [40, 50]
   [70, 80, 90]
True
False
False


In [ ]:
#-----------B+ Tree-----Actual records/data are kept at the leaf level. The internal nodes mainly act as navigation/indexes.
"""
Imagine we have these patient IDs:
10, 20, 30, 40, 50, 60, 70

We arrange them like this:
                 [30 | 50]              >>  The top [30 | 50] helps us find where to go.The bottom boxes contain the actual records.
                /    |    \
               /     |     \
       [10,20]   [30,40]   [50,60,70]
          ↓          ↓           ↓
       actual      actual       actual
        data        data         data

          [10,20] → [30,40] → [50,60,70]
"""
class BPlusNode:
    def __init__(self, keys, leaf=False):
        self.keys = keys
        self.children = []
        self.leaf = leaf
        self.next = None

"""
| Variable   | Simple meaning                 |
| ---------- | ------------------------------ |
| `keys`     | Numbers/data inside the box    |
| `children` | Other boxes connected below it |
| `leaf`     | Is this box at the bottom?     |
| `next`     | Link to the next bottom box    |

root.children[0]  → left
root.children[1]  → middle
root.children[2]  → right

"""

left = BPlusNode([10, 20], leaf=True)         # << - we are creating a box - [10, 20] . Leaf = True means "This is a bottom-level box.
middle = BPlusNode([30, 40], leaf=True)
right = BPlusNode([50, 60, 70], leaf=True)

left.next = middle                            # << - The left box should point to the middle box. [10,20] ─────→ [30,40]
middle.next = right                           # << - [10,20] ─────→ [30,40] ─────→ [50,60,70]

root = BPlusNode([30, 50])                    # << - This is top box 
root.children = [left, middle, right]         # << -- store those 3 child nodes

def search(root, value):
    if value < root.keys[0]:
        current = root.children[0]
    elif value < root.keys[1]:
        current = root.children[1]
    else:
        current = root.children[2]
    return value in current.keys

print(search(root, 40))

True
